Christakakis, P., Pechlivani, E.-M., & Dimou, P. (2025). Thermal PV Panel Detection and Fault Detection Dataset for UAV-Based Inspection [Data set]. Zenodo. https://doi.org/10.5281/zenodo.16420123
(CC-BY)

# YOLOv8 Training
Download and prepare data

In [ ]:
# Replace with the actual Zenodo URL
!wget -O data.zip "https://zenodo.org/records/16420123/files/Thermal%20PV%20Panel%20Detection%20Dataset%20for%20UAV%20Inspection.zip?download=1"
!unzip data.zip

In [ ]:
import os

data_source_dir = '/content/data_source'

# List all files in the directory
all_files = os.listdir(data_source_dir)

# Separate image and text files
txt_files = {f for f in all_files if f.endswith('.txt')}
image_extensions = ['.jpg', '.jpeg', '.png']
image_files = {f for f in all_files if any(f.lower().endswith(ext) for ext in image_extensions)}

# Get the base names without extensions for comparison
txt_basenames = {os.path.splitext(f)[0] for f in txt_files}
image_basenames = {os.path.splitext(f)[0] for f in image_files}

# Find image basenames that do not have a corresponding text basename
extra_image_basenames = image_basenames - txt_basenames

# Find the full filenames of the extra image files
extra_image_files = [f for f in image_files if os.path.splitext(f)[0] in extra_image_basenames]

print("Image files without a corresponding .txt file:")
for extra_image in extra_image_files:
    print(extra_image)

Image files without a corresponding .txt file:
DJI_20230213170209_0001_T_JPG.rf.95b9dbd9c1b1989c443bc23d004cb355.jpg
DJI_20230213170213_0003_T_JPG.rf.18f02dbe2b7e0d7981d3578a9243d566.jpg


In [ ]:
import os
import shutil

data_source_dir = '/content/data_source'
destination_dir = '/content/'

for file_name in extra_image_files:
    source_file_path = os.path.join(data_source_dir, file_name)
    destination_file_path = os.path.join(destination_dir, file_name)
    try:
        shutil.move(source_file_path, destination_file_path)
        print(f"Moved {file_name} to {destination_dir}")
    except FileNotFoundError:
        print(f"Error: {file_name} not found in {data_source_dir}")
    except Exception as e:
        print(f"Error moving {file_name}: {e}")

Moved DJI_20230213170209_0001_T_JPG.rf.95b9dbd9c1b1989c443bc23d004cb355.jpg to /content/
Moved DJI_20230213170213_0003_T_JPG.rf.18f02dbe2b7e0d7981d3578a9243d566.jpg to /content/


# Prepare training data for YOLO v8+
Prepare training data for YOLOv8 using the ultralytics package from the data in the "data_source" directory, splitting it into 70% for training and 30% for validation.

## Install ultralytics

### Subtask:
Install the necessary `ultralytics` package.


**Reasoning**:
The subtask is to install the `ultralytics` package. I will use pip to install the package.



In [ ]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.2 MB/s eta 0:00:00


## Create directory structure

### Subtask:
Create the required directory structure for YOLOv8 training data (e.g., `train/images`, `train/labels`, `val/images`, `val/labels`).


**Reasoning**:
Create the directory structure required for YOLOv8 training data.



In [ ]:
import os

# 1. Define the base directory for the YOLOv8 dataset
yolov8_dataset_dir = '/content/yolov8_dataset'

# 2. Create the base directory
os.makedirs(yolov8_dataset_dir, exist_ok=True)

# 3. Within the base directory, create the train directory
train_dir = os.path.join(yolov8_dataset_dir, 'train')
os.makedirs(train_dir, exist_ok=True)

# 4. Within the train directory, create the images and labels directories
os.makedirs(os.path.join(train_dir, 'images'), exist_ok=True)
os.makedirs(os.path.join(train_dir, 'labels'), exist_ok=True)

# 5. Within the base directory, create the val directory
val_dir = os.path.join(yolov8_dataset_dir, 'val')
os.makedirs(val_dir, exist_ok=True)

# 6. Within the val directory, create the images and labels directories
os.makedirs(os.path.join(val_dir, 'images'), exist_ok=True)
os.makedirs(os.path.join(val_dir, 'labels'), exist_ok=True)

print(f"Created YOLOv8 dataset directory structure at {yolov8_dataset_dir}")

Created YOLOv8 dataset directory structure at /content/yolov8_dataset


## Split data

### Subtask:
Split the images and corresponding label files from `data_source` into the `train` and `val` directories based on a 70/30 ratio.


**Reasoning**:
List the image files, shuffle them, split them into training and validation sets, and then move the images and their corresponding label files to the respective train and val directories.



In [ ]:
import os
import shutil
import random

data_source_dir = '/content/data_source'
yolov8_dataset_dir = '/content/yolov8_dataset'

# List all image files in the data_source directory
image_extensions = ['.jpg', '.jpeg', '.png']
all_image_files = [f for f in os.listdir(data_source_dir) if any(f.lower().endswith(ext) for ext in image_extensions)]

# Shuffle the list of image files randomly
random.shuffle(all_image_files)

# Calculate the number of images for the training set (70% of the total)
train_size = int(len(all_image_files) * 0.7)

# Split the shuffled list of image files into training and validation sets
train_image_files = all_image_files[:train_size]
val_image_files = all_image_files[train_size:]

# Function to move files
def move_files(file_list, destination_base_dir):
    for image_file in file_list:
        # Construct the full path for the current image in data_source
        source_image_path = os.path.join(data_source_dir, image_file)

        # Construct the corresponding label file name (replace image extension with .txt)
        label_file = os.path.splitext(image_file)[0] + '.txt'

        # Construct the full path for the current label file in data_source
        source_label_path = os.path.join(data_source_dir, label_file)

        # Construct the destination path for the image
        destination_image_path = os.path.join(yolov8_dataset_dir, destination_base_dir, 'images', image_file)

        # Construct the destination path for the label file
        destination_label_path = os.path.join(yolov8_dataset_dir, destination_base_dir, 'labels', label_file)

        # Move the image file
        try:
            shutil.move(source_image_path, destination_image_path)
            # print(f"Moved image {image_file} to {destination_image_path}") # Optional: uncomment for verbose output
        except FileNotFoundError:
            print(f"Error: Image file {image_file} not found.")
        except Exception as e:
            print(f"Error moving image file {image_file}: {e}")

        # Move the label file
        try:
            shutil.move(source_label_path, destination_label_path)
            # print(f"Moved label {label_file} to {destination_label_path}") # Optional: uncomment for verbose output
        except FileNotFoundError:
             print(f"Error: Label file {label_file} not found.")
        except Exception as e:
            print(f"Error moving label file {label_file}: {e}")


# Move training files
print("Moving training files...")
move_files(train_image_files, 'train')

# Move validation files
print("Moving validation files...")
move_files(val_image_files, 'val')

print("Data splitting and moving complete.")

Moving training files...
Moving validation files...
Data splitting and moving complete.


## Create data yaml

### Subtask:
Create a YAML file that defines the dataset path, the number of classes, and the class names.


**Reasoning**:
Create a dictionary containing the dataset path, the number of classes, and the class names, and then write this dictionary to a YAML file.



In [ ]:
import yaml
import os

# Define the dataset path
dataset_path = '/content/yolov8_dataset'

# Define the class names
class_names = ['Panel'] # Assuming 'Panel' is the only class based on previous analysis

# Define the number of classes
num_classes = len(class_names)

# Create the dictionary
data_yaml_content = {
    'path': dataset_path,
    'train': 'train/images',  # Path to training images relative to 'path'
    'val': 'val/images',      # Path to validation images relative to 'path'
    'nc': num_classes,
    'names': class_names
}

# Define the path for the data.yaml file
data_yaml_path = os.path.join(dataset_path, 'data.yaml')

# Write the dictionary to the data.yaml file
with open(data_yaml_path, 'w') as file:
    yaml.dump(data_yaml_content, file, default_flow_style=False)

print(f"Created data.yaml file at: {data_yaml_path}")

Created data.yaml file at: /content/yolov8_dataset/data.yaml


## Train YOLOv8n model

In [ ]:
from ultralytics import YOLO

# Load a YOLOv8n model
model = YOLO('yolov8n.pt')  # Load a pretrained nano model

# Define the path to your data.yaml file
data_yaml_path = '/content/yolov8_dataset/data.yaml'

# Train the model
results = model.train(data=data_yaml_path, epochs=500, imgsz=640)

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolov8_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True,

## Validate YOLOv8n model

In [ ]:
# Validate the model
results = model.val()

# Display the results
print(results)

Ultralytics 8.3.222 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1516.9±362.2 MB/s, size: 40.6 KB)
val: Scanning /content/yolov8_dataset/val/labels.cache... 75 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 75/75 130.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 0.5it/s 10.2s
                   all         75       7330      0.992      0.921      0.957      0.853
Speed: 5.6ms preprocess, 4.3ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /content/runs/detect/val2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f54cbe99f10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Prec

In [ ]:
# Access and print detailed metrics from the results object
print("Detailed Validation Metrics:")
print(f"  Metrics at 50% IOU (mAP@50): {results.results_dict['metrics/mAP50(B)']: .4f}")
print(f"  Metrics at 50-95% IOU (mAP@50-95): {results.results_dict['metrics/mAP50-95(B)']: .4f}")
print(f"  Precision: {results.results_dict['metrics/precision(B)']: .4f}")
print(f"  Recall: {results.results_dict['metrics/recall(B)']: .4f}")

# You can explore other attributes of the results object for more details
# print(results.speed) # Inference speed
print(results.confusion_matrix) # Confusion matrix (if generated)


Detailed Validation Metrics:
  Metrics at 50% IOU (mAP@50):  0.9571
  Metrics at 50-95% IOU (mAP@50-95):  0.8529
  Precision:  0.9925
  Recall:  0.9214


![results.png](./results.png)

![confusion_matrix.png](confusion_matrix.png)

![confusion_matrix_normalized.png](confusion_matrix_normalized.png)

![BoxF1_curve.png](BoxF1_curve.png)

![train_batch2.jpg](train_batch2.jpg)

![val_batch1_pred.jpg](val_batch1_pred.jpg)

![val_batch2_pred.jpg](val_batch2_pred.jpg)